# Dashboard Data Prep (Google Sheets → Looker Studio)

This notebook builds every sheet the Looker Studio dashboard reads from, in a single run. It replaces an earlier version that had grown into ~20 sheets through trial and error — several of them duplicates, a couple mislabeled, and two (`WAF`'s and `Demographics`' age columns) filled with `np.random.randint()` placeholder data that never got replaced with the real computation. That last one mattered: those random ages are almost certainly why a couple of the dashboard's bubble charts showed an oddly narrow, meaningless age clustering around ~44 (the midpoint of the random range).

This version:
- Reuses the same `Country`/`EdLevel` cleanup and age-bucket mapping as `02_Data_Wrangling.ipynb`, instead of a third, slightly different mapping
- Computes every "most common age" the same way, once, correctly — no random fallback data anywhere
- Produces exactly the 12 tables the dashboard's panels actually need, each built once
- Runs as one pass: no manually commenting/uncommenting cells between runs

**A note on data lineage:** the standalone `survey_data_updated_clean.xlsx` workbook delivered alongside this notebook was computed directly from a smaller, separate 18,845-row export (the raw data embedded in the originally-uploaded `survey_data_updated.xlsx`) rather than from this repo's 65,437-row wrangled dataset. The two are independently real and internally consistent, but not the same sample — counts in the xlsx will not match what this notebook produces if run against `survey_data_clean.csv`. Re-run this notebook once the pipeline's cleaned dataset is in the repo to bring the live dashboard onto the same 65,437-row dataset as every other notebook.

**Run this in Google Colab** (it needs `google.colab.auth` for the Google Sheets write step). Update `DATA_URL` below to point at the raw GitHub URL for `data/processed/survey_data_clean.csv` once it's pushed, or swap in a Google Drive path — both options are in the cell.


## 1. Setup

In [ ]:
!pip install -q gspread gspread-dataframe google-auth oauth2client

In [ ]:
import pandas as pd
import numpy as np

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default

## 2. Load the cleaned dataset

In [ ]:
# Option A (recommended): read straight from the GitHub repo once 02_Data_Wrangling.ipynb's
# output has been pushed, so this notebook and the rest of the project share one source of truth.
DATA_URL = "https://raw.githubusercontent.com/<your-username>/<your-repo>/main/data/processed/survey_data_clean.csv"

# Option B: if you'd rather not depend on GitHub being up to date, mount Drive instead and
# point this at wherever you saved survey_data_clean.csv:
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_URL = "/content/drive/MyDrive/capstone/survey_data_clean.csv"

df = pd.read_csv(DATA_URL)
df.shape

## 3. Shared cleanup (same mappings as the wrangling notebook)

In [ ]:
age_mapping = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70,
}

country_mapping = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "UK",
    "Russian Federation": "Russia",
    "Iran, Islamic Republic of...": "Iran",
    "Venezuela, Bolivarian Republic of...": "Venezuela",
    "Hong Kong (S.A.R.)": "Hong Kong",
    "Congo, Republic of the...": "Congo",
    "Micronesia, Federated States of...": "Micronesia",
    "C\u00f4te d\'Ivoire": "Cote d\'Ivoire",
    "Republic of Korea": "South Korea",
    "Korea, Republic of": "South Korea",
}

edlevel_mapping = {
    "Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)": "Bachelor\'s",
    "Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master\'s",
    "Some college/university study without earning a degree": "Some college",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary school",
    "Associate degree (A.A., A.S., etc.)": "Associate",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional",
    "Primary/elementary school": "Primary school",
    "Something else": "Other",
}

df["Age_numeric"] = df["Age"].map(age_mapping)
df["Country_clean"] = df["Country"].replace(country_mapping)
df["EdLevel_clean"] = df["EdLevel"].replace(edlevel_mapping)

## 4. Helper functions

Three small functions cover all 12 output tables — no copy-pasted per-category cells.

In [ ]:
def top_n_counts(dataframe, column, n=10):
    """Top-N value counts for a semicolon-delimited multi-select column."""
    counts = (
        dataframe[column]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
        .head(n)
        .reset_index()
    )
    counts.columns = [column.replace("HaveWorkedWith", "").replace("Admired", ""), "Count"]
    return counts


def top_n_with_age(dataframe, column, n=10):
    """Top-N counts plus each item's average respondent age.

    Note: an earlier version of this used the *mode* of the age bracket, labeled
    MostCommonAge. That turned out to be a bad metric here \u2014 with 25-34 year-olds
    making up 41% of all respondents, the mode collapses to the same bracket for
    almost every single language/database/framework, so it doesn't actually
    differentiate anything (every bubble in a bubble chart would land at the same
    height). The mean does differentiate, and the pattern is a real one: newer
    frameworks (Solid.js, Astro, NestJS) skew younger, legacy ones (jQuery, ASP.NET,
    Drupal) skew notably older.
    """
    exploded = (
        dataframe[[column, "Age_numeric"]]
        .dropna()
        .assign(Item=lambda x: x[column].str.split(";"))
        .explode("Item")
        .assign(Item=lambda x: x["Item"].str.strip())
    )

    counts = exploded.groupby("Item").size().reset_index(name="Count")
    avg_age = (
        exploded.groupby("Item")["Age_numeric"]
        .mean()
        .round(1)
        .reset_index(name="AverageAge")
    )

    result = counts.merge(avg_age, on="Item").sort_values("Count", ascending=False).head(n)
    return result.reset_index(drop=True)


def top_n_with_share(dataframe, column, n=10):
    """Top-N counts plus each item's share of all mentions (for the treemap panel)."""
    counts = (
        dataframe[column]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
    )
    total = counts.sum()
    result = counts.head(n).reset_index()
    result.columns = [column.replace("Admired", ""), "Count"]
    result["Percentage"] = (result["Count"] / total * 100).round(2)
    return result

## 5. Build every table the dashboard needs

**Current Technology Usage:**

In [ ]:
Top_Languages = top_n_counts(df, "LanguageHaveWorkedWith")
Top_Databases = top_n_counts(df, "DatabaseHaveWorkedWith")
Top_Platforms = top_n_counts(df, "PlatformHaveWorkedWith")
Top_WebFrameworks_Age = top_n_with_age(df, "WebframeHaveWorkedWith")

**Future Technology Trends** (using the `*Admired` columns, i.e. what respondents want to keep using / start using):

In [ ]:
Top_LanguagesAdmired_Age = top_n_with_age(df, "LanguageAdmired")
Top_DatabasesAdmired_Age = top_n_with_age(df, "DatabaseAdmired")
Top_PlatformsAdmired = top_n_with_share(df, "PlatformAdmired")
Top_WebframesAdmired_Age = top_n_with_age(df, "WebframeAdmired")

**Demographics:**

In [ ]:
age_counts = df["Age"].value_counts()
Demographics_Age = age_counts.reset_index()
Demographics_Age.columns = ["Age", "Count"]
Demographics_Age["Percentage"] = (Demographics_Age["Count"] / age_counts.sum() * 100).round(1)

Demographics_Country = (
    df["Country_clean"].value_counts().reset_index()
)
Demographics_Country.columns = ["Country", "Count"]

Demographics_Education = (
    df["EdLevel_clean"].value_counts().reset_index()
)
Demographics_Education.columns = ["EdLevel", "Count"]

Demographics_Age_Education = (
    pd.crosstab(df["Age"], df["EdLevel_clean"])
    .reset_index()
)

## 6. Push everything to Google Sheets

One loop instead of one copy-pasted block per sheet — each sheet is deleted and recreated if it already exists, so this is safe to re-run after any change above.

In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_NAME = "survey_data_updated"  # change if your sheet has a different name
sh = gc.open(SPREADSHEET_NAME)

sheets_to_write = {
    "Top_Languages": Top_Languages,
    "Top_Databases": Top_Databases,
    "Top_Platforms": Top_Platforms,
    "Top_WebFrameworks_Age": Top_WebFrameworks_Age,
    "Top_LanguagesAdmired_Age": Top_LanguagesAdmired_Age,
    "Top_DatabasesAdmired_Age": Top_DatabasesAdmired_Age,
    "Top_PlatformsAdmired": Top_PlatformsAdmired,
    "Top_WebframesAdmired_Age": Top_WebframesAdmired_Age,
    "Demographics_Age": Demographics_Age,
    "Demographics_Country": Demographics_Country,
    "Demographics_Education": Demographics_Education,
    "Demographics_Age_Education": Demographics_Age_Education,
}

existing_titles = {ws.title for ws in sh.worksheets()}

for name, frame in sheets_to_write.items():
    if name in existing_titles:
        sh.del_worksheet(sh.worksheet(name))
    worksheet = sh.add_worksheet(title=name, rows=len(frame) + 1, cols=len(frame.columns))
    set_with_dataframe(worksheet, frame)
    print(f"{name}: {frame.shape[0]} rows written")

print("\nDone \u2014 all 12 sheets rebuilt.")

## 7. Sheets this replaces

If you're cleaning up the actual Google Sheet, these can be deleted once the 12 sheets above are confirmed working in Looker Studio:

| Old sheet(s) | Replaced by | Why |
|---|---|---|
| `Languages` | — (deleted) | Mislabeled duplicate of `Top_Platforms` |
| `PAF`, `DAF`, `WAF` | `Top_PlatformsAdmired`, `Top_DatabasesAdmired_Age`, `Top_WebframesAdmired_Age` | Raw unaggregated intermediate frames (36K–53K rows each) that leaked into the sheet instead of their aggregated top-10 versions; `WAF` also had the random-age bug |
| `Admired_Language/Database/Platform/Webframe` | `Top_*Admired_Age` / `Top_PlatformsAdmired` | Had their own corrupted `MostCommonAge` values (some outside the valid 16–70 range) |
| `Admired_Tech_Age`, `Top_LanguageAdmired_Age`, `Top_PlatformAdmired_Age`, `Top_WebframeAdmired_Age`, `Top_DatabaseAdmired_Age` | `Top_*Admired_Age` / `Top_PlatformsAdmired` | Duplicate top-10s of the same data, split across too many near-identical sheets |
| `Top_Databases_Age` | `Top_DatabasesAdmired_Age` (usage version is `Top_WebFrameworks_Age`'s sibling, not built here — add a `top_n_with_age(df, "DatabaseHaveWorkedWith")` call if the dashboard needs a *current-usage* age breakdown for databases too) | Superseded by consistent naming |
| `Demographics` | `Demographics_Age` + `Demographics_Country` + `Demographics_Education` + `Demographics_Age_Education` | Split into one focused table per chart; dropped the random `Age_rand` column |

**One thing to check on your end:** after re-running this, open the Looker Studio charts that used the old `WAF`/`Demographics` age data (the Desired Web Frameworks bubble chart, and the age line on the Education-by-level chart) and re-point them at the new sheets — the age values will look different now because they're real.